# Lab 5: TF-IDF, cosine similarity, and Word2Vec

Guidelines: https://github.com/MohammedMosuily/NLP_Lab5

Run cells from top to bottom. Dataset paths are relative to this folder.

## Task 1: Cosine similarity

In [1]:
import re
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import Word2Vec

sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]
vectors = TfidfVectorizer().fit_transform(sentences)
pd.DataFrame(cosine_similarity(vectors), index=sentences, columns=sentences)

,This is the first document.,This document is the second document.,And this is the third one.,Is this the first document?
This is the first document.,1.000000,0.646926,0.307772,1.000000
This document is the second document.,0.646926,1.000000,0.225240,0.646926
And this is the third one.,0.307772,0.225240,1.000000,0.307772
Is this the first document?,1.000000,0.646926,0.307772,1.000000


## Task 2: Important words with TF-IDF
Larger weights identify more distinctive terms in each sentence. Remove English stopwords to focus on content words.

In [2]:
documents = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data",
]
tfidf = TfidfVectorizer(stop_words="english")
vectors = tfidf.fit_transform(documents)
weights = pd.DataFrame(vectors.toarray(), columns=tfidf.get_feature_names_out())
print(weights)
for i, row in weights.iterrows():
    print("Document", i + 1, row[row > 0].sort_values(ascending=False).to_dict())

    analyze      best   courses      data    fields  important   science  \
0  0.000000  0.000000  0.000000  0.273526  0.463121   0.463121  0.704430   
1  0.000000  0.584483  0.584483  0.345205  0.000000   0.000000  0.444514   
2  0.542701  0.000000  0.000000  0.641055  0.000000   0.000000  0.000000   

   scientists  
0    0.000000  
1    0.000000  
2    0.542701  
Document 1 {'science': 0.7044302447025257, 'fields': 0.46312055911762967, 'important': 0.46312055911762967, 'data': 0.27352646269326086}
Document 2 {'best': 0.5844829010200651, 'courses': 0.5844829010200651, 'science': 0.444514311537431, 'data': 0.34520501686496574}
Document 3 {'data': 0.6410554491745127, 'analyze': 0.5427006131762078, 'scientists': 0.5427006131762078}


## Task 3: Train skip-gram on Simpsons dialogue
Use the supplied `spoken_words` column and the cleaning procedure from the guidelines.

In [3]:
df = pd.read_csv("dataset/simpsons_script_lines.csv", low_memory=False)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

tokens = [clean_text(text).split() for text in df["spoken_words"].dropna().astype(str)]
tokens = [sentence for sentence in tokens if sentence]
model = Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1, workers=1, seed=42)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


## Task 4: Similar words

In [4]:
for word in ["homer", "marge", "bart"]:
    print(word, model.wv.most_similar(word))

homer [('marge', 0.8806141018867493), ('abe', 0.8706626892089844), ('bart', 0.845809280872345), ('abraham', 0.8337423801422119), ('eliza', 0.8317188024520874), ('bartholomew', 0.8284366130828857), ('barney', 0.8254926800727844), ('monty', 0.8248452544212341), ('grampa', 0.8202494978904724), ('ralph', 0.8172776699066162)]
marge [('abe', 0.8885388374328613), ('homer', 0.8806141018867493), ('sweetie', 0.8521940112113953), ('lisa', 0.8507775068283081), ('eliza', 0.8492140173912048), ('maude', 0.84917813539505), ('fellas', 0.8462461829185486), ('honey', 0.8459284901618958), ('becky', 0.8458413481712341), ('sweetheart', 0.8411526083946228)]
bart [('lisa', 0.8864496946334839), ('milhouse', 0.8658613562583923), ('abe', 0.8607434034347534), ('eliza', 0.8538054823875427), ('homer', 0.8458093404769897), ('ralph', 0.8365848064422607), ('apu', 0.8350124359130859), ('grampa', 0.8306716680526733), ('grandma', 0.8303202986717224), ('marge', 0.826081395149231)]


## Task 5: Odd words out
These answers reflect the learned dialogue contexts; they need not match character relationships.

In [5]:
for words in [["jimbo", "milhouse", "kearney"], ["nelson", "bart", "milhouse"], ["homer", "patty", "selma"]]:
    print(words, "->", model.wv.doesnt_match(words))

['jimbo', 'milhouse', 'kearney'] -> kearney
['nelson', 'bart', 'milhouse'] -> nelson
['homer', 'patty', 'selma'] -> homer
